# Quack Flamegraph — Design Spec

**Status:** draft for review  
**Date:** 2026-08-21  
**Epic:** [issue #1](https://github.com/kevintruong/quack-flamegraph/issues/1)  
**Repo:** DuckDB Rust C-API extension template (`rusty_quack` → rename to `quack_flamegraph`)

This notebook is the authoritative spec. Formal gates are native `ns_mermaid` flowchart cells (profile `relational_lia` v1). The `architecture` Mermaid profile is **not** implemented for proof; architecture drawings below are presentation flowcharts, and the routing/invariants are the `@spec` cells that follow.

## Solver evidence (already run)

| Claim | Tool | Result | `solve_id` |
|---|---|---|---|
| Unique v1 approach is `stacks_vtab` | `solve_constraints` | `sat` model `approach=stacks_vtab` | `sol_ee629dd86dfa4518` |
| No other approach satisfies sample hard gates | `solve_constraints` + `not stacks` | `unsat` | `sol_34ee9e35b8824539` |
| Feature model: duckdb-rs + stacks, tree optional off | `solve_rules` configuration | `sat`+`pass` | `sol_a8d47081e4f54d3f` |
| Node unique / parent FK / inclusive identity | `solve_rules` data_integrity | `sat`+`pass` | `sol_39496bef664e4289` |
| Nested-set sibling order | `data_integrity.temporal_consistency` | `sat`+`pass` | `sol_baf571e354164983` |
| Child width inside parent | `layout.containment` | `sat`+`pass` | `sol_37e4d9d44437471a` |
| Sibling slices disjoint | `layout.non_overlap` | `sat`+`pass` | `sol_46e746016b8a47ae` |

## Reference sample (decision input)

Turso graph-index capture, 2026-08-21:

- File: `/tmp/spur-turso-flamegraph.PaowCw/folded-demangled.txt`
- 12,918 folded lines, **5,788 nonzero** stacks, **67,517** samples, 15 MB
- 7,130 zero-count parent frames (must drop)
- Queries that localized the hotspot: exclusive leaf `GROUP BY`, inclusive `unnest(frames)`, `contains()` buckets
- Inclusive-by-name is **sample coverage**, not tree `SUM(inclusive)`
- Wall ~118 s, on-CPU dominant


## Goals

- Load a CPU/allocation profile into DuckDB and query it as tables.
- Answer optimization questions with ordinary SQL: hottest **exclusive** frames, hottest **inclusive** (sample-coverage) frames, domain buckets, later subtree and diffs.
- Ship as a loadable DuckDB community-style extension on the existing Rust C-API template.
- Parser, not the SQL user, owns folded-format hazards (last-field count, zero-sample parent lines, `module\`symbol` names).

## Non-goals

- Rendering SVG flamegraphs (use inferno / Brendan Gregg / samply UI).
- Replacing `cargo flamegraph` / samply capture.
- Live attach to a running process.
- Unbounded recursive CTE as the product (the extension materializes relations).
- Claiming the `architecture` Mermaid profile is a proof (unavailable).

## Decision

**Primary relation is folded stacks**, not nested-interval nodes.

| Wave | Ship |
|---|---|
| **v1** | `read_folded` VTab on **duckdb-rs**: `frames VARCHAR[]`, `samples BIGINT`; macros `flamegraph_exclusive` / `flamegraph_inclusive` |
| **later** | `flamegraph_nodes` (nested intervals), `read_pprof`, replacement scan `FROM 'cpu.folded'`, quack-rs **only if** replacement/copy cannot be registered on duckdb-rs |


## Architecture (presentation)

Not an NS-Spec proof. The `architecture` / C4 profile cannot verify. Binding contracts are `@spec ARCH-*` cells after this diagram.

```mermaid
flowchart TB
  subgraph Capture["Capture - out of band"]
    PERF["perf / samply / xctrace / cargo-flamegraph"]
    FOLDED["*.folded Brendan Gregg\nmodule`symbol;... count"]
    PPROF["*.pb.gz pprof Profile proto"]
    PERF --> FOLDED
    PERF --> PPROF
  end

  subgraph Ext["quack_flamegraph.duckdb_extension"]
    EP["duckdb_entrypoint_c_api\nregister VTab + macros"]

    subgraph V1["v1 - duckdb-rs"]
      RF["read_folded(path) VTab"]
      PARSE["parse line\nlast field = count\nsplit on semicolon"]
      ZERO["drop samples = 0"]
      STACKS["stacks relation\nframes VARCHAR[]\nsamples BIGINT"]
      MX["flamegraph_exclusive(src)"]
      MI["flamegraph_inclusive(src)"]
      RF --> PARSE --> ZERO --> STACKS
      STACKS --> MX
      STACKS --> MI
    end

    subgraph Later["later"]
      RP["read_pprof(path) VTab"]
      INTERN["intern function / location / string_table"]
      TOSTACK["emit same stacks schema"]
      NODES["flamegraph_nodes(src)\ntrie-merge stacks\nlft rgt parent_id\nexclusive inclusive x_offset"]
      REPL["replacement scan\nFROM 'cpu.folded'\nFROM 'cpu.pb.gz'"]
      DIFF["flamegraph_diff(a, b)"]
      RP --> INTERN --> TOSTACK
      STACKS -.-> NODES
      TOSTACK -.-> NODES
      REPL --> RF
      REPL --> RP
      NODES --> DIFF
    end

    subgraph SDK["SDK gate"]
      DRS["duckdb-rs C-API\nVScalar + VTab\nthis template"]
      QRS["quack-rs\nonly if replacement/copy\ncannot register on duckdb-rs"]
    end
    EP --> DRS
    DRS -.-> QRS
  end

  subgraph SQL["SQL consumer"]
    EX["exclusive: list_extract(frames, -1) GROUP BY"]
    IN["inclusive: unnest(frames) GROUP BY\n= sample coverage"]
    BU["buckets: contains(leaf, 'ts_')"]
    DE["descendants: c.lft BETWEEN p.lft AND p.rgt"]
  end

  FOLDED --> RF
  PPROF --> RP
  MX --> EX
  MI --> IN
  STACKS --> BU
  NODES --> DE
```

**Data flow.** Capture tools stay outside the extension. v1 only reads folded text and yields one stack relation. Later ingest (pprof) **converges on the same stacks schema**, then optionally trie-merges to nodes. Replacement scan is syntax sugar over the same table functions. Diff is a node- or stack-keyed full outer join, not a third encoding.


## Formal gates

Profile pin: `relational_lia` / header `flowchart`, version 1. Each cell is one `@spec`. Run the cell to publish proof badges. Architecture-policy proofs are not claimed.

### ARCH-PHASE — v1 vs later slices

Only `stacks_vtab` is v1. Nested nodes, pprof, replacement scan, and quack-rs are `later`.

In [ ]:
flowchart TD
    SPEC["`@spec ARCH-PHASE
@type Slice = enum[stacks_vtab, nodes_vtab, pprof, replacement, quack_sdk]
@type Wave = enum[v1, later]
@input slice: Slice
@output status: Wave
@requires PRE: true`"]
    V1["`@branch V1_STACKS
@when slice = stacks_vtab
@ensures V1_WAVE: status = v1`"]
    N["`@branch LATER_NODES
@when slice = nodes_vtab
@ensures N_WAVE: status = later`"]
    P["`@branch LATER_PPROF
@when slice = pprof
@ensures P_WAVE: status = later`"]
    R["`@branch LATER_REPL
@when slice = replacement
@ensures R_WAVE: status = later`"]
    Q["`@branch LATER_QUACK
@when slice = quack_sdk
@ensures Q_WAVE: status = later`"]
    CHECK["`@verify DET: prove determinism
@verify COVER: prove partition_coverage
@verify EXCL: prove partition_exclusive
@verify STATUSES: witness each status`"]
    SPEC --> V1 --> CHECK
    SPEC --> N --> CHECK
    SPEC --> P --> CHECK
    SPEC --> R --> CHECK
    SPEC --> Q --> CHECK

### ARCH-INGEST — format to table function

Folded → `read_folded`. pprof → `read_pprof` (later). Anything else → reject. Replacement scan rewrites filename to one of these functions; it is not a third parser.

In [ ]:
flowchart TD
    SPEC["`@spec ARCH-INGEST
@type Format = enum[folded, pprof, unknown]
@type Fn = enum[read_folded, read_pprof, reject]
@input format: Format
@output status: Fn
@requires PRE: true`"]
    FOLDED["`@branch FOLDED
@when format = folded
@ensures FOLDED_FN: status = read_folded`"]
    PPROF["`@branch PPROF
@when format = pprof
@ensures PPROF_FN: status = read_pprof`"]
    UNKNOWN["`@branch UNKNOWN
@when format = unknown
@ensures UNKNOWN_FN: status = reject`"]
    CHECK["`@verify DET: prove determinism
@verify COVER: prove partition_coverage
@verify EXCL: prove partition_exclusive
@verify STATUSES: witness each status`"]
    SPEC --> FOLDED --> CHECK
    SPEC --> PPROF --> CHECK
    SPEC --> UNKNOWN --> CHECK

### ARCH-QUERY — which relation serves which question

Exclusive-by-leaf and inclusive sample-coverage read **stacks**. Descendants of a call-path node read **nodes**. Do not compute inclusive-by-name as `SUM(node.inclusive)` — that double-counts nested callees.

In [ ]:
flowchart TD
    SPEC["`@spec ARCH-QUERY
@type Query = enum[exclusive_leaf, inclusive_coverage, descendants]
@type Rel = enum[stacks, nodes]
@input query: Query
@output status: Rel
@requires PRE: true`"]
    EX["`@branch EXCLUSIVE
@when query = exclusive_leaf
@ensures EX_REL: status = stacks`"]
    IN["`@branch INCLUSIVE
@when query = inclusive_coverage
@ensures IN_REL: status = stacks`"]
    DE["`@branch DESCENDANTS
@when query = descendants
@ensures DE_REL: status = nodes`"]
    CHECK["`@verify DET: prove determinism
@verify COVER: prove partition_coverage
@verify EXCL: prove partition_exclusive
@verify STATUSES: witness each status`"]
    SPEC --> EX --> CHECK
    SPEC --> IN --> CHECK
    SPEC --> DE --> CHECK

### ZERO-FILTER — parser owns zero-count parent frames

The Turso folded file is 55% zero-sample lines (xctrace/sample collapse parents). `read_folded` drops `samples = 0` before emitting rows. Domain here is a bounded Int stand-in (`0..100`); the implementation uses `BIGINT >= 0`.

In [ ]:
flowchart TD
    SPEC["`@spec ZERO-FILTER
@type Keep = enum[keep, drop]
@input samples: Int
@output status: Keep
@requires GE0: samples >= 0
@requires LE: samples <= 100`"]
    DROP["`@branch DROP
@when samples = 0
@ensures DROP_S: status = drop`"]
    KEEP["`@branch KEEP
@when samples > 0
@ensures KEEP_S: status = keep`"]
    CHECK["`@verify DET: prove determinism
@verify COVER: prove partition_coverage
@verify EXCL: prove partition_exclusive
@verify STATUSES: witness each status`"]
    SPEC --> DROP --> CHECK
    SPEC --> KEEP --> CHECK

### NODE-BALANCE — later `flamegraph_nodes` identity

When stacks are trie-merged, each node satisfies `inclusive = exclusive + sum(child.inclusive)`. Fixture from the earlier catalog solve: leaf `10 = 10 + 0`, root `30 = 0 + 30`.

In [ ]:
flowchart TD
    SPEC["`@spec NODE-BALANCE
@input exclusive: Int
@input child_sum: Int
@input inclusive: Int
@requires EX_GE0: exclusive >= 0
@requires CH_GE0: child_sum >= 0
@requires EX_LE: exclusive <= 100
@requires CH_LE: child_sum <= 100
@requires IN_LE: inclusive <= 200
@requires BALANCE: inclusive = exclusive + child_sum
@requires IN_GE_EX: inclusive >= exclusive`"]
    CHECK["`@verify BAL: witness consistency`"]
    SPEC --> CHECK

### ARCH-SDK — stay on duckdb-rs until replacement scan forces a move

v1 table functions stay on this template. If DuckDB C-API replacement scans cannot be registered from duckdb-rs, that **later** slice may use quack-rs. Do not switch SDK for v1 VTabs.

In [ ]:
flowchart TD
    SPEC["`@spec ARCH-SDK
@type Cap = enum[vtab_only, need_replacement]
@type Sdk = enum[duckdb_rs, quack_rs]
@input cap: Cap
@output status: Sdk
@requires PRE: true`"]
    VTAB["`@branch VTAB
@when cap = vtab_only
@ensures VTAB_SDK: status = duckdb_rs`"]
    REPL["`@branch REPL
@when cap = need_replacement
@ensures REPL_SDK: status = quack_rs`"]
    CHECK["`@verify DET: prove determinism
@verify COVER: prove partition_coverage
@verify EXCL: prove partition_exclusive
@verify STATUSES: witness each status`"]
    SPEC --> VTAB --> CHECK
    SPEC --> REPL --> CHECK

## v1 schema and SQL

### `read_folded(path VARCHAR)` → stacks

| Column | Type | Notes |
|---|---|---|
| `frames` | `VARCHAR[]` | Root → leaf order, `;` split. Keep `module\`symbol` text. |
| `samples` | `BIGINT` | Strictly `> 0` |
| `source` | `VARCHAR` | Absolute or given path (optional, useful for UNION) |

Parser rules:

1. Each line: `stack[;stack...] <count>` — count is the last space-separated field.
2. Strip optional Instruments thread prefix (`Thread_…:`).
3. Drop lines with `count = 0`.
4. Reject lines with no count or negative count.
5. Empty file → empty table, not an error.

### Macros (SQL, registered from the extension)

```sql
-- exclusive self time (Turso query 1)
SELECT list_extract(frames, -1) AS leaf, SUM(samples) AS exclusive
FROM read_folded('cpu.folded')
GROUP BY 1
ORDER BY 2 DESC;

-- inclusive sample coverage (Turso query 2)
SELECT unnest(frames) AS frame, SUM(samples) AS inclusive
FROM read_folded('cpu.folded')
GROUP BY 1
ORDER BY 2 DESC;

-- domain bucket (Turso query 3)
SELECT SUM(samples) FILTER (WHERE list_extract(frames, -1) LIKE '%ts_%') AS tree_sitter_self
FROM read_folded('cpu.folded');
```

Macros `flamegraph_exclusive(src)` and `flamegraph_inclusive(src)` wrap the first two. Named table functions or `CREATE MACRO` via the C API — whichever duckdb-rs exposes; both must be SQLLogic-tested.

### Crate layout (v1)

```
src/lib.rs              entrypoint, register
src/folded.rs           line parser + VTab bind/init/func
src/macros.rs           exclusive / inclusive SQL macros
test/sql/read_folded.test
test/sql/exclusive.test
test/data/tiny.folded   including zero-count parent lines
```

Rename package `rusty_quack` → `quack_flamegraph` when first landing functions. Keep `cdylib`, `duckdb_entrypoint_c_api`, `make debug` footer, SQLLogic tests.


## Later slices (in spec, not v1 implementation)

All later ingest **converges on stacks**, then optionally derives nodes. Do not invent a third encoding per format.

### `flamegraph_nodes(src)` — nested intervals

Trie-merge identical prefixes of `frames`. Assign DFS `lft`/`rgt`, `parent_id` (NULL at roots), `depth`, `exclusive`, `inclusive`, `x_offset` (sample-axis start).

| Column | Type |
|---|---|
| `node_id` | `BIGINT` |
| `parent_id` | `BIGINT` NULL |
| `name` | `VARCHAR` |
| `depth` | `INTEGER` |
| `exclusive` | `BIGINT` |
| `inclusive` | `BIGINT` |
| `lft` | `INTEGER` |
| `rgt` | `INTEGER` |
| `x_offset` | `BIGINT` |

Invariants (catalog-proved on a finite fixture):

- unique `node_id`
- `parent_id` FK MATCH SIMPLE (root parent absent)
- `inclusive(n) = exclusive(n) + Σ inclusive(child)`
- `lft < rgt`; sibling intervals finish-start
- child x-range contained in parent; siblings non-overlapping

```sql
-- descendants of one call-path node
SELECT c.name, c.exclusive, c.inclusive, c.depth
FROM flamegraph_nodes('cpu.folded') AS p
JOIN flamegraph_nodes('cpu.folded') AS c
  ON c.lft BETWEEN p.lft AND p.rgt
WHERE p.node_id = 2;
```

`SUM(inclusive) GROUP BY name` is **not** a supported inclusive-by-name query (ARCH-QUERY).

### `read_pprof(path)`

Parse gzip `perftools.profiles.Profile`. Walk `sample.location_id[]` → `Location.line[]` → `Function.name` via `string_table`. Default value column: `sample.value[default_sample_type]` (else index 0). Emit the **same** `frames`/`samples` schema as `read_folded`. Inline frames inside one Location expand in leaf-to-root or root-to-leaf consistently with folded (root → leaf in `frames`).

### Replacement scan

`FROM 'cpu.folded'` → `read_folded('cpu.folded')`.  
`FROM 'cpu.pb.gz'` / `*.pprof` → `read_pprof(...)`.  
Unknown suffix → do not claim the file (return no replacement).

If duckdb-rs cannot register replacement scans, implement this slice with quack-rs (`ARCH-SDK`) without rewriting v1 VTabs.

### `flamegraph_diff(a, b)`

Full outer join on `frames` (or on node `name`+`depth` path). Columns: `samples_a`, `samples_b`, `delta`. Stacks-level diff is enough for the Turso-style exclusive/inclusive questions; node-level diff is optional after `flamegraph_nodes`.

### quack-rs

Not a second product. Trigger: C-API feature missing from duckdb-rs (replacement scan, copy, named params). v1 remains duckdb-rs on this template.


## Errors

| Condition | Behavior |
|---|---|
| Missing file | IO error from DuckDB, not empty table |
| Malformed folded line | skip with `tracing`/`stderr` warning **or** fail the scan — pick **fail the scan** (strict). Tests cover one bad line. |
| Negative count | fail the scan |
| Empty file | empty relation |
| Truncated pprof gzip | fail `read_pprof` |

## Testing

SQLLogicTest under `test/sql/`:

- `read_folded` on `tiny.folded` including a zero-count parent: zeros absent, remaining samples sum to the nonzero total.
- Exclusive leaf order for a 3-stack fixture (`main;foo 10` / `main;bar 20`).
- Inclusive unnest: `main` = 30, `foo` = 10, `bar` = 20.
- Bad line → error.
- Golden: subset of `/tmp/spur-turso-flamegraph.PaowCw/folded-demangled.txt` (or a checked-in 20-stack excerpt) — top exclusive leaf is `ts_node_child_iterator_next` **only** if the excerpt preserves that; otherwise use the tiny fixture.

Later: pprof round-trip from a tiny proto, node inclusive identity, replacement scan.

## Task boundaries (for writing-plans)

| Task | Files | Depends |
|---|---|---|
| Rename crate + entrypoint | `Cargo.toml`, `src/lib.rs`, `Makefile`, tests | — |
| Folded parser unit tests | `src/folded.rs` | rename |
| `read_folded` VTab | `src/lib.rs`, `src/folded.rs` | parser |
| Exclusive/inclusive macros | `src/macros.rs` | VTab |
| SQLLogic tests + tiny fixture | `test/sql/*` | VTab + macros |
| (later) trie + `flamegraph_nodes` | `src/nodes.rs` | stacks |
| (later) pprof | `src/pprof.rs` | stacks schema |
| (later) replacement scan | entrypoint | folded + pprof |
| (later) quack-rs only if blocked | new crate path | replacement |

Parser and VTab can parallelize after rename if the parser is a pure function.

## Risks

- Folded names contain spaces (rare); count = last field, not `rsplitn` on first space from the left.
- Rust generic names are huge; `VARCHAR[]` is fine at 6k stacks / 15 MB.
- LTO/ICF mis-names in the **profile**, not in this extension — we store strings as given.
- `architecture` Mermaid profile cannot prove topology; do not treat the presentation diagram as verified.

## Open for user review

Spec is written. Formal `@spec` ids: `ARCH-PHASE`, `ARCH-INGEST`, `ARCH-QUERY`, `ZERO-FILTER`, `NODE-BALANCE`, `ARCH-SDK`. Please review this notebook (diagram + proof badges after cells are run) before the implementation plan.
